In [ ]:
# Import necessary libraries
import pandas as pd
import re
import nltk
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
from warnings import filterwarnings
filterwarnings('ignore')

pd.set_option('display.max_columns', None)

In [ ]:
# Load the dataset
raw_data = pd.read_csv('Data/postings.csv')
df = raw_data.copy()
df.head()

In [ ]:
# Explore the dataset
print(f"Rows: {df.shape[0]}")
print("-"*20)
print(f"\nColumns: {df.shape[1]}")
print("-"*20)
print(f"\nColumns: {df.columns.tolist()}")

In [ ]:
df.info()

In [ ]:
# Fix data types
count_cols = ['company_id', 'views', 'applies', 'remote_allowed']
for col in count_cols:
    df[col] = df[col].astype('Int64')

time_cols = ['original_listed_time', 'expiry', 'closed_time', 'listed_time']
for col in time_cols:
    df[col] = pd.to_datetime(df[col], unit='ms', errors='coerce')

df.info()

In [ ]:
df.sample(10)

In [ ]:
# Standardize column names
df.columns = df.columns.str.strip().str.lower().str.replace(' ', '_')
df.sample(10)

In [ ]:
# check for duplicates
df.duplicated().sum()

In [ ]:
df[df.duplicated(subset=['job_id'], keep=False)]

In [ ]:
# Check for missing values
df.isnull().sum()

## Handle missing values

In [ ]:
df['company_name'] = df['company_name'].fillna('Unknown')

df['company_id'] = df['company_id'].astype('object').fillna('Unknown').astype(str)

# Drop null values from description
df.dropna(subset=['description'], inplace=True)

df['remote_allowed'] = df['remote_allowed'].fillna(0)
df['applies'] = df['applies'].fillna(0)
df['views'] = df['views'].fillna(0)

df['formatted_experience_level'] = df['formatted_experience_level'].fillna('Not Specified')
df['posting_domain'] = df['posting_domain'].fillna('Unknown')

df['has_salary_listed'] = df['normalized_salary'].notna().astype(int)

In [ ]:
columns_to_drop = [
        'skills_desc', 
        'closed_time', 
        'fips', 
        'zip_code',
        'max_salary', 
        'min_salary', 
        'med_salary', 
        'pay_period', 
        'currency', 
        'compensation_type',
        'job_posting_url',
        'application_url'
    ]
    
df.drop(columns=columns_to_drop, errors='ignore', inplace=True)

In [ ]:
# Date features
df['listing_year'] = df['listed_time'].dt.year
df['listing_month'] = df['listed_time'].dt.month
df['listing_day'] = df['listed_time'].dt.day

In [ ]:
# cleaning description column

from typing import Any


import re
import nltk
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer

stop_words = set[Any](stopwords.words('english'))
lemmatizer = WordNetLemmatizer()

def nlp_cleaner(text):
    if not isinstance(text, str):
        return ""

    # Remove HTML tags
    text = re.sub(r'<.*?>', ' ', text)

     # Remove URLs
    text = re.sub(r'http\S+|www\S+', ' ', text)

    # Remove email addresses
    text = re.sub(r'\S+@\S+', ' ', text)  
    
    # Remove newlines and carriage returns
    text = re.sub(r'[\n\r]', ' ', text)

    # Remove special characters (keep only alphanumeric)
    text = re.sub(r'[^a-z0-9 ]', ' ', text) 



    # text.split() handles tokenization AND naturally drops multiple spaces (\s+)
    cleaned_words = [
        lemmatizer.lemmatize(word) for word in text.split() if word not in stop_words
    ]
    
    # Rejoin the tokens into a single clean string
    return " ".join(cleaned_words)

# Apply it to the dataframe
df['clean_description'] = df['description'].apply(nlp_cleaner)

In [ ]:
# Exteact skills
skills = [
    'python',
    'sql',
    'power bi',
    'tableau',
    'excel',
    'aws',
    'azure',
    'gcp',
    'machine learning',
    'deep learning',
    'pandas',
    'numpy',
    'spark',
    'hadoop'
]

def extract_skills(text):

    found = []

    text = str(text).lower()

    for skill in skills:
        if skill in text:
            found.append(skill)

    return found

df['skills'] = (
    df['clean_description']
    .apply(extract_skills)
)


df['skill_count'] = (
    df['skills']
    .apply(len)
)

In [ ]:
df.sample(50)

In [ ]:
import pandas as pd

job_skills = pd.read_csv('Data/jobs/job_skills.csv')
skills = pd.read_csv('Data/mappings/skills.csv')
job_industries = pd.read_csv('Data/jobs/job_industries.csv')
industries = pd.read_csv('Data/mappings/industries.csv')
companies = pd.read_csv('Data/companies/companies.csv')


display(job_skills)
print("-"*20)
display(skills)
print("-"*20)
display(job_industries)
print("-"*20)
display(industries)
print("-"*20)
display(companies)

In [ ]:
datasets = {
    "Job Skills": job_skills,
    "Skills Mapping": skills,
    "Job Industries": job_industries,
    "Industries Mapping": industries,
    "Companies": companies
}

def generate_quality_report(data_dict):
    report = []
    
    for name, df in data_dict.items():
        total_rows = len(df)
        total_cols = len(df.columns)
        duplicates = df.duplicated().sum()
        
        # missing values 
        null_counts = df.isnull().sum()
        missing_cols = [f"{col} ({count})" for col, count in null_counts.items() if count > 0]
        missing_str = ", ".join(missing_cols) if missing_cols else "No Missing Data"
        
        # Append to report
        report.append({
            "Dataset Name": name,
            "Rows": f"{total_rows:,}",
            "Columns": total_cols,
            "Duplicate Rows": f"{duplicates:,}",
            "Columns w/ Nulls (Count)": missing_str
        })
        
    return pd.DataFrame(report)

# 3. Generate and display the clean summary
print("--- DATA QUALITY REPORT ---")
dq_report = generate_quality_report(datasets)
display(dq_report)

In [ ]:
for name, df in datasets.items():
    print(f"\n{'='*40}")
    print(f" DATASET: {name.upper()}")
    print(f"{'='*40}")
    
    # Display the standard info
    df.info()
    
    print("-" * 40)
    display(df.head(2))

In [ ]:
merged_skills = job_skills.merge(skills, on='skill_abr', how='left')
skills_aggregated = merged_skills.groupby('job_id')['skill_name'].apply(
    lambda x: ', '.join(x.dropna().unique())
).reset_index()
skills_aggregated.rename(columns={'skill_name': 'explicit_required_skills'}, inplace=True)

industries['industry_name'] = industries['industry_name'].fillna('Unknown Industry')
merged_industries = job_industries.merge(industries, on='industry_id', how='left')
industries_aggregated = merged_industries.groupby('job_id')['industry_name'].apply(
    lambda x: ', '.join(x.unique())
).reset_index()

companies['name'] = companies['name'].fillna('Confidential Company')
companies['description'] = companies['description'].fillna('')
companies.rename(columns={'description': 'company_description'}, inplace=True)


companies['company_size'] = companies['company_size'].fillna('Not Specified').astype(str)

# Clean up web scraping location artifacts
companies['state'] = companies['state'].replace(['0', '-'], 'Not Specified').fillna('Not Specified')
companies['city'] = companies['city'].replace(['0', '-'], 'Not Specified').fillna('Not Specified')


In [ ]:
display(skills_aggregated.head(3))
print("-"*20)
display(industries_aggregated.head(3))
print("-"*20)
display(companies.head(3))

In [ ]:
salaries = pd.read_csv('Data/jobs/salaries.csv')
benefits = pd.read_csv('Data/jobs/benefits.csv')
comp_specialities = pd.read_csv('Data/companies/company_specialities.csv')
comp_industries = pd.read_csv('Data/companies/company_industries.csv')
employee_counts = pd.read_csv('Data/companies/employee_counts.csv')

display(salaries)
print("-"*20)
display(benefits)
print("-"*20)
display(comp_specialities)
print("-"*20)
display(comp_industries)
print("-"*20)
display(employee_counts)

In [ ]:
raw_metadata_datasets = {
    "Raw Salaries": salaries,
    "Raw Benefits": benefits,
    "Raw Company Specialities": comp_specialities,
    "Raw Company Industries": comp_industries,
    "Raw Employee Counts": employee_counts
}

metadata_dq_report = generate_quality_report(raw_metadata_datasets)
display(metadata_dq_report)

print("\n1. Salaries:")
display(salaries.head(3))

print("\n2. Benefits:")
display(benefits.head(3))

print("\n3. Company Specialities:")
display(comp_specialities.head(3))

print("\n4. Company Industries:")
display(comp_industries.head(3))

print("\n5. Employee Counts:")
display(employee_counts.head(3))